In [3]:
# ============================================================
# HOME CREDIT DEFAULT RISK
# GLOBAL EDA FOR RELATIONAL DATASET
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0. PATH CONFIG
# ------------------------------------------------------------

cwd = Path.cwd()

# Nếu notebook chạy từ /notebooks
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())
# ------------------------------------------------------------
# 1. LOAD DATASETS
# ------------------------------------------------------------

FILES = {
    "application_train": "application_train.csv",
    "application_test": "application_test.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "installments_payments": "installments_payments.csv",
    "POS_CASH_balance": "POS_CASH_balance.csv",
    "credit_card_balance": "credit_card_balance.csv",
}

datasets = {}

for name, filename in FILES.items():
    path = DATA_DIR / filename

    if path.exists():
        print(f"Loading {filename} ...")
        datasets[name] = pd.read_csv(path)
    else:
        print(f"[WARNING] Missing file: {filename}")

print("\nLoaded datasets:")
for name, df in datasets.items():
    print(f"{name:25s}: {df.shape}")

PROJECT_ROOT: c:\Users\NHAT THU\KLTN\Personalized-Federated-Heterogeneous-Graph-Learning-for-Credit-Risk-Prediction-
DATA_DIR: c:\Users\NHAT THU\KLTN\Personalized-Federated-Heterogeneous-Graph-Learning-for-Credit-Risk-Prediction-\data\raw
DATA_DIR exists: True
Loading application_train.csv ...
Loading application_test.csv ...
Loading bureau.csv ...
Loading bureau_balance.csv ...
Loading previous_application.csv ...
Loading installments_payments.csv ...
Loading POS_CASH_balance.csv ...
Loading credit_card_balance.csv ...

Loaded datasets:
application_train        : (307511, 122)
application_test         : (48744, 121)
bureau                   : (1716428, 17)
bureau_balance           : (27299925, 3)
previous_application     : (1670214, 37)
installments_payments    : (13605401, 8)
POS_CASH_balance         : (10001358, 8)
credit_card_balance      : (3840312, 23)


DATASETS OVERVIEW

In [4]:
# ------------------------------------------------------------
# 2. DATASET OVERVIEW
# ------------------------------------------------------------

overview_rows = []

for name, df in datasets.items():

    overview_rows.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "numeric_cols": df.select_dtypes(include=np.number).shape[1],
        "categorical_cols": df.select_dtypes(exclude=np.number).shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
        "missing_pct": df.isna().mean().mean() * 100
    })

overview = pd.DataFrame(overview_rows)

overview

,table,rows,columns,numeric_cols,categorical_cols,duplicate_rows,missing_cells,missing_pct
0,application_train,307511,122,106,16,0,9152465,24.395942
1,application_test,48744,121,105,16,0,1404419,23.811687
2,bureau,1716428,17,14,3,0,3939947,13.502552
3,bureau_balance,27299925,3,2,1,0,0,0.000000
4,previous_application,1670214,37,21,16,0,11109336,17.976877
5,installments_payments,13605401,8,8,0,0,5810,0.005338
6,POS_CASH_balance,10001358,8,7,1,0,52158,0.065189
7,credit_card_balance,3840312,23,22,1,0,5877356,6.654074


Home Credit có 3 key quan trọng:
SK_ID_CURR
SK_ID_PREV
SK_ID_BUREAU
-> Check 2 key trên

In [5]:
# ------------------------------------------------------------
# 3. KEY / CARDINALITY CHECK
# ------------------------------------------------------------

KEYS = ["SK_ID_CURR", "SK_ID_PREV", "SK_ID_BUREAU"]

key_report = []

for table_name, df in datasets.items():

    for key in KEYS:

        if key in df.columns:

            key_report.append({
                "table": table_name,
                "key": key,
                "rows": len(df),
                "unique_values": df[key].nunique(),
                "missing": df[key].isna().sum(),
                "duplicate_occurrences": len(df) - df[key].nunique()
            })

key_report = pd.DataFrame(key_report)

key_report

,table,key,rows,unique_values,missing,duplicate_occurrences
0,application_train,SK_ID_CURR,307511,307511,0,0
1,application_test,SK_ID_CURR,48744,48744,0,0
2,bureau,SK_ID_CURR,1716428,305811,0,1410617
3,bureau,SK_ID_BUREAU,1716428,1716428,0,0
4,bureau_balance,SK_ID_BUREAU,27299925,817395,0,26482530
5,previous_application,SK_ID_CURR,1670214,338857,0,1331357
6,previous_application,SK_ID_PREV,1670214,1670214,0,0
7,installments_payments,SK_ID_CURR,13605401,339587,0,13265814
8,installments_payments,SK_ID_PREV,13605401,997752,0,12607649
9,POS_CASH_balance,SK_ID_CURR,10001358,337252,0,9664106


DATA QUALITY CHO TỪNG BẢNG

In [6]:
# ------------------------------------------------------------
# 4. GENERAL DATA QUALITY
# ------------------------------------------------------------

def data_quality_report(df, table_name):

    print("=" * 80)
    print(table_name.upper())
    print("=" * 80)

    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())

    print("\nData types:")
    print(df.dtypes.value_counts())

    missing = (
        df.isna()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
    )

    print("\nTop missing columns:")
    print(missing.head(15))

    return missing


missing_reports = {}

for name, df in datasets.items():
    missing_reports[name] = data_quality_report(df, name)

APPLICATION_TRAIN
Shape: (307511, 122)
Duplicate rows: 0

Data types:
float64    65
int64      41
str        16
Name: count, dtype: int64

Top missing columns:
COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
COMMONAREA_MEDI             69.872297
NONLIVINGAPARTMENTS_MEDI    69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MEDI       68.354953
LIVINGAPARTMENTS_MODE       68.354953
FLOORSMIN_MODE              67.848630
FLOORSMIN_AVG               67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_AVG             66.497784
YEARS_BUILD_MODE            66.497784
dtype: float64
APPLICATION_TEST
Shape: (48744, 121)
Duplicate rows: 0

Data types:
float64    65
int64      40
str        16
Name: count, dtype: int64

Top missing columns:
COMMONAREA_AVG              68.716150
COMMONAREA_MEDI             68.716150
COMMONAREA_MODE      

TARGET EDA - chỉ có application train có target(nghĩa là kết quả có trả được nợ hay ko)

In [8]:
# ------------------------------------------------------------
# 5. TARGET DISTRIBUTION
# ------------------------------------------------------------

# Keep this cell runnable even after restarting the notebook kernel.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if "datasets" not in globals():
    datasets = {}

if "application_train" not in datasets:
    datasets["application_train"] = pd.read_csv(
        project_root / "data" / "raw" / "application_train.csv"
    )

app_train = datasets["application_train"]

target_counts = app_train["TARGET"].value_counts().sort_index()
target_pct = (
    app_train["TARGET"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print(target_counts)

print("\nPercentage:")
print(target_pct)

fig, ax = plt.subplots(figsize=(6, 4))

target_counts.plot(
    kind="bar",
    ax=ax
)

ax.set_title("TARGET Distribution")
ax.set_xlabel("TARGET")
ax.set_ylabel("Count")

for i, value in enumerate(target_counts.values):
    ax.text(
        i,
        value,
        f"{target_pct.iloc[i]:.2f}%",
        ha="center",
        va="bottom"
    )

plt.show()

NameError: name 'datasets' is not defined

In [5]:
# ------------------------------------------------------------
# 5. TARGET DISTRIBUTION
# ------------------------------------------------------------

# Keep this cell runnable even after restarting the notebook kernel.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if "datasets" not in globals():
    datasets = {}

if "application_train" not in datasets:
    datasets["application_train"] = pd.read_csv(
        project_root / "data" / "raw" / "application_train.csv"
    )

app_train = datasets["application_train"]

target_counts = app_train["TARGET"].value_counts().sort_index()
target_pct = (
    app_train["TARGET"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print(target_counts)

print("\nPercentage:")
print(target_pct)

fig, ax = plt.subplots(figsize=(6, 4))

target_counts.plot(
    kind="bar",
    ax=ax
)

ax.set_title("TARGET Distribution")
ax.set_xlabel("TARGET")
ax.set_ylabel("Count")

for i, value in enumerate(target_counts.values):
    ax.text(
        i,
        value,
        f"{target_pct.iloc[i]:.2f}%",
        ha="center",
        va="bottom"
    )

plt.show()

NameError: name 'datasets' is not defined

 Missingness của application_train

In [1]:
# ------------------------------------------------------------
# 6. APPLICATION TRAIN MISSINGNESS
# ------------------------------------------------------------

missing_app = (
    app_train
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(missing_app.head(30))

top_missing = missing_app.head(30)

fig, ax = plt.subplots(figsize=(10, 8))

ax.barh(
    top_missing.index[::-1],
    top_missing.values[::-1]
)

ax.set_xlabel("Missing (%)")
ax.set_title("Top 30 Missing Features - application_train")

plt.tight_layout()
plt.show()

NameError: name 'app_train' is not defined